# 段位期別の安定性と時系列評価の成立条件
P1 → P2 → P3 の順に実行します。元のC5完了キャッシュだけを読み、結果を新しいフォルダに保存します。

**判定基準（事前固定）**　低段位期と高段位期を別々に判定します。主分析は非重複10対局、感度分析は20対局。各群の窓を時系列順・ほぼ同数で前中後に三分します。有効20窓以上、欠損20%以下、機会数中央値5以上、各期有効5窓以上を要求します。一定値、率で端点（0または1）が3窓以上あり全て分母1～2の場合は判断保留です。端点は有効窓に対する割合です。

割合18指標は同じ群・幅の測定条件を満たした割合指標のSD中央値・IQR中央値以下かを確認します。平均型2指標はそれぞれ同群の1対局SD・IQRの半分以下かを確認し、基準が0の場合は保留します。さらに3期間の中央値の範囲がその群のIQR以下のとき「安定」とします。片群でも判断保留なら総合判定は判断困難。機会数基準3・10も参考に計算します。全期間の記述用判定を将来のモデル選択に直接流用しません。

In [ ]:
# P1　Drive接続・固定コードの取得・合成データによる動作確認
from google.colab import drive
from pathlib import Path
import importlib.util
import os
import subprocess
import sys
import tempfile
import urllib.request

drive.mount('/content/drive')
COMMIT = '884add7b670b702c1c834a226add132f846db873'
source_root = Path(tempfile.mkdtemp(prefix='mahjong_smoothing_'))
code_dir = source_root / 'stability'
base = f'https://raw.githubusercontent.com/yuukienomoto/report_-syobon_n/{COMMIT}/research_v2/'
for name in ('mahjong_research/__init__.py', 'mahjong_research/tenhou.py',
             'mahjong_research/analysis.py', 'mahjong_research/inputs.py', 'mahjong_research/legacy.py',
             'smoothing/study.py', 'smoothing/presentation.py', 'smoothing/test_smoothing.py',
             'stability/stability.py', 'stability/test_stability.py'):
    dst = source_root / name
    dst.parent.mkdir(exist_ok=True)
    with urllib.request.urlopen(base + name, timeout=60) as response:
        dst.write_bytes(response.read())
deps = {'pandas':'pandas', 'numpy':'numpy', 'matplotlib':'matplotlib', 'sklearn':'scikit-learn'}
missing = [pkg for module, pkg in deps.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
env = dict(os.environ, PYTHONPATH=os.pathsep.join(map(str,[source_root, source_root/'smoothing'])), MPLBACKEND='Agg')
subprocess.run([sys.executable, '-m', 'unittest', '-q', 'test_stability'], cwd=code_dir, env=env, check=True)
print('合成データの動作確認が完了しました。')

In [ ]:
# P2　過去のC3・C5で使った実行フォルダを選択（候補が1件なら自動）
ROOT = Path('/content/drive/MyDrive/卒論')
RUN_DIR = None  # 例: ROOT / 'research_v2_runs' / '実行フォルダ名'
OUTPUT_ROOT = ROOT / 'research_v2_stability'
if RUN_DIR is None:
    choices = sorted(p.parent for p in (ROOT/'research_v2_runs').glob('*/input_plan.json')
                     if (p.parent/'shared_games.csv').is_file())
    if len(choices) != 1:
        print(*choices, sep='\n')
        raise ValueError('ROOTを確認し、C5が完了した実行フォルダをRUN_DIRに指定してP2を再実行してください。')
    RUN_DIR = choices[0]
RUN_DIR = Path(RUN_DIR)
print('既存の入力:', RUN_DIR, '\n新しい出力の親:', OUTPUT_ROOT)

In [ ]:
# P3　安定性分析・3分割の成立監査・保存
from IPython.display import Markdown, display
import pandas as pd
runner = 'import sys; from stability import run; print(run(sys.argv[1],sys.argv[2]))'
proc = subprocess.run([sys.executable, '-c', runner, str(RUN_DIR), str(OUTPUT_ROOT)],
                      cwd=code_dir, env=env, stdout=subprocess.PIPE, text=True, check=True)
RESULT = Path(proc.stdout.strip().splitlines()[-1])
print('保存先:', RESULT)
display(Markdown((RESULT/'report_ja.md').read_text(encoding='utf-8')))
for name in ['feature_decisions','group_stability','fold_feasibility']:
    display(pd.read_csv(RESULT/(name+'.csv')))

**結果の読み方**　主分析の候補は `primary_candidates.csv`、全項目の採否理由は `feature_decisions.csv`、判定に用いた実測値と基準値は `group_stability.csv` です。前・中・後の集計は `period_statistics.csv`、機会数基準3・5・10の比較は `threshold_sensitivity.csv` に保存します。

`fold_feasibility.csv` は既存の時間境界を用いた学習／評価群別の窓数と成立条件です。境界をまたぐ窓は除外し、そのIDも `fold_membership.csv` に記録します。集計は低段位期と高段位期のどちらが識別できたかを示すものではありません。本Notebookは予測スコアを生成しません。